In [5]:
# 📦 Install if not already done
# pip install -U langchain langchain-google-genai langchain-community langchain-text-splitters faiss-cpu python-dotenv

# 1️⃣ Imports
import os
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain.tools import tool
from langchain.agents import create_agent          # v1-native replacement for initialize_agent/AgentExecutor
from langgraph.checkpoint.memory import InMemorySaver  # v1-native replacement for ConversationBufferMemory

# 2️⃣ Load API key
load_dotenv(".env")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# 3️⃣ Setup LLM (Gemini)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Run this once in a cell to create a sample file for testing
sample_text = """LangChain is a framework for developing applications powered by large language models (LLMs).
It provides tools for chaining together LLM calls, integrating retrieval-augmented generation (RAG),
connecting to vector stores, and building agents that can use external tools.
LangChain was created by Harrison Chase and was first released in October 2022.
It has since grown into a widely used framework for building LLM-powered applications,
including chatbots, question-answering systems, and autonomous agents.
"""

with open("sample.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)
    
# 4️⃣ Create Vector DB (Retriever)
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)

embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 5️⃣ Retrieval as a plain tool (RetrievalQA chain is removed in v1 —
# the agent's own LLM now reasons over the retrieved passages instead of
# a separate QA chain calling a second LLM internally)
@tool
def langchain_retriever(query: str) -> str:
    """Use this to answer questions about LangChain framework, features, or its creator."""
    docs = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in docs)

tools = [langchain_retriever]

# 6️⃣ Memory: InMemorySaver checkpointer, addressed by thread_id
# (replaces ConversationBufferMemory; persists across .invoke() calls
# as long as you reuse the same thread_id)
checkpointer = InMemorySaver()

# 7️⃣ Build the agent (replaces initialize_agent + AgentExecutor)
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools when needed to answer accurately.",
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "user-1"}}

def ask(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
    return result["messages"][-1].content

# 8️⃣ Ask Questions (RAG-Style)
print("1️⃣ First Question")
print("Answer:", ask("What is LangChain?"))

print("\n2️⃣ Follow-up")
print("Answer:", ask("Who created it?"))

print("\n3️⃣ Combined Reasoning")
print("Answer:", ask("Explain LangChain's use in AI workflows."))

C:\Users\Manikandan\AppData\Local\Temp\ipykernel_12536\35491700.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


1️⃣ First Question
Answer: LangChain is a framework designed to facilitate the development of applications powered by large language models (LLMs). It offers tools for various functionalities, including chaining LLM calls, integrating retrieval-augmented generation (RAG), connecting to vector stores, and constructing agents capable of utilizing external tools.

Harrison Chase created LangChain, which was initially released in October 2022. Since then, it has become a popular framework for building LLM-powered applications such as chatbots, question-answering systems, and autonomous agents.

2️⃣ Follow-up
Answer: LangChain was created by Harrison Chase.

3️⃣ Combined Reasoning
Answer: LangChain is a framework designed to facilitate the development of applications powered by large language models (LLMs) within AI workflows. It offers several key functionalities:

*   **Chaining LLM Calls:** It provides tools to link multiple LLM calls together, allowing for more complex and multi-step re

In [ ]:
#!pip install langchain langchain-google-genai langchain-community langchain-text-splitters faiss-cpu python-dotenv


In [ ]:
#!pip install -U langchain langchain-google-genai langchain-community langchain-text-splitters faiss-cpu python-dotenv